# RFM 客户分层

客户标识必须用 `customer_unique_id`。`customer_id` 是订单级匿名 ID，一单一个，用它算频次永远是 1。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
rfm = query("""
WITH base AS (
    SELECT customer_unique_id, order_id, purchase_ts, price
    FROM dws_order_wide
    WHERE order_status = 'delivered'
),
agg AS (
    SELECT customer_unique_id,
           COUNT(DISTINCT order_id) AS frequency,
           SUM(price) AS monetary,
           MAX(purchase_ts) AS last_ts
    FROM base
    GROUP BY customer_unique_id
)
SELECT customer_unique_id,
       frequency,
       ROUND(monetary, 2) AS monetary,
       DATEDIFF((SELECT MAX(purchase_ts) FROM base), last_ts) AS recency_days
FROM agg
""")
print("客户数", len(rfm))
rfm.describe().round(1)

客户数 93358


,frequency,monetary,recency_days
count,93358.0,93358.0,93358.0
mean,1.0,141.6,237.5
std,0.2,215.7,152.6
min,1.0,0.8,0.0
25%,1.0,47.6,114.0
50%,1.0,89.7,218.0
75%,1.0,154.7,346.0
max,15.0,13440.0,713.0


97% 的客户只买过一次，频次这一维几乎没有区分度，所以后面只用 R 和 M。

In [3]:
print("只买一次的客户占比", round((rfm.frequency == 1).mean()*100, 2), "%")

只买一次的客户占比 97.0 %


In [4]:
import numpy as np

df = rfm.copy()
# 最近买过 = 高分；买得晚 = 低分
df["R"] = pd.qcut(df.recency_days, 4, labels=[4, 3, 2, 1]).astype(int)
df["M"] = pd.qcut(df.monetary, 4, labels=[1, 2, 3, 4]).astype(int)

def seg(row):
    if row.R >= 3 and row.M >= 3:
        return "核心高价值"
    if row.R <= 2 and row.M >= 3:
        return "流失高客单"
    if row.R >= 3:
        return "新客潜力"
    return "沉睡低价值"

df["segment"] = df.apply(seg, axis=1)
df.segment.value_counts()

segment
核心高价值    23600
沉睡低价值    23599
新客潜力     23080
流失高客单    23079
Name: count, dtype: int64

In [5]:
summary = df.groupby("segment").agg(
    人数=("customer_unique_id", "count"),
    平均R=("recency_days", "mean"),
    平均M=("monetary", "mean"),
    贡献收入=("monetary", "sum"),
).round(1)
summary["人数占比%"] = (summary.人数 / len(df) * 100).round(1)
summary["收入占比%"] = (summary.贡献收入 / df.monetary.sum() * 100).round(1)
summary.sort_values("贡献收入", ascending=False)

,人数,平均R,平均M,贡献收入,人数占比%,收入占比%
segment,,,,,,
核心高价值,23600,111.6,235.4,5556234.7,25.3,42.0
流失高客单,23079,363.5,236.5,5458999.9,24.7,41.3
沉睡低价值,23599,363.8,47.4,1117618.5,25.3,8.5
新客潜力,23080,111.0,47.2,1088645.1,24.7,8.2
